# Laboratory 11 — Statistical ensembles

In this laboratory you find out where the Boltzmann factor comes from, and you find it without
simulating anything. A small system shares energy quanta with a bath of Einstein oscillators;
the computer lists every way the whole composite can hold its quanta, gives each one the same
weight — module 08's postulate — and asks how often the small system is found in each state.

The route: the model and its objects (Part 1); a composite small enough to list by hand
(Part 2); the flat joint distribution and the curved marginal (Part 3); growing the bath at a
fixed temperature (Part 4); measuring how fast the finite bath converges, and the term the
Boltzmann derivation drops (Part 5); reading the temperature off the bath (Part 6); whose
temperature it is (Part 7); the two-level system in closed form (Part 8); an optional coda on
the atmosphere (Part 9); the automated checks (Part 10); and free exploration (Part 11).

Work through it in order. Where the notebook asks you to predict, write your prediction in the
cell provided **before** running the next cell. A prediction you have committed to is the only
reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | a small system — a two-level system with a gap of a whole number of quanta, or an Einstein solid of a few oscillators — sharing $q_{\mathrm{tot}}$ quanta with an Einstein-solid bath of $N_{\mathrm{bath}}$ oscillators |
| **Dynamics** | none: every accessible joint microstate is counted and weighted equally; no time, no sampling, no random numbers |
| **Boundary** | the composite is isolated, $q_{\mathrm{tot}}$ exactly fixed; the wall passes quanta only, so no work crosses it |
| **Ensemble** | exactly microcanonical for the composite; the system's marginal approaches the canonical form as $N_{\mathrm{bath}}$ grows |
| **Ignored** | the wall's coupling energy, unequal quantum sizes, spatial structure, exchange rates |
| **Valid when** | system and bath share one quantum size and the composite is isolated; the canonical form also needs $E_s \ll U_{\mathrm{bath}}$ |
| **Failure modes** | a bath too small to have a smooth $\ln\Omega$ slope; $E_s$ comparable to the total energy, where the marginal departs from the exponential |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import ensembles
from thermolab.constants import AMU, K_B
from thermolab.validation import relative_error, scaling_exponent

# This module draws no random numbers at all: the enumeration is exact, so there is no
# generator to pass and nothing for a seed to change.

QUANTUM = 1.0e-21  # J, the size of one energy quantum, shared by system and bath
two_level = ensembles.two_level(1, QUANTUM)  # ground at 0, excited one quantum up
solid = ensembles.einstein_levels(3, 30, QUANTUM)  # three oscillators, levels k = 0..30
print(f"one quantum = {QUANTUM:.1e} J = {QUANTUM / K_B:.1f} K x k_B")
print(f"a bath with one quantum per oscillator sits at "
      f"T = {ensembles.bath_temperature(1.0, QUANTUM):.2f} K")

## Part 1 — The objects

A **level list** gives each of the small system's energies, in whole quanta, and how many
microstates share it. The two-level system has two levels of one microstate each. The
three-oscillator solid holds $k$ quanta in $g_k = \binom{k + 2}{k}$ ways — module 01's stars
and bars — so its levels are degenerate.

The bath is an Einstein solid of $N_{\mathrm{bath}}$ oscillators. Holding $q$ quanta, it has
$\Omega_{\mathrm{bath}} = \binom{q + N_{\mathrm{bath}} - 1}{q}$ microstates.

In [ ]:
print("two-level system:   quanta", two_level.quanta, "  degeneracy", two_level.degeneracy)
print("three-oscillator solid, first six levels:")
print("   quanta    ", solid.quanta[:6])
print("   degeneracy", np.rint(solid.degeneracy[:6]).astype(int))

### Predict

Commit to an answer for each of the module page's four predictions before going further —
they are what this laboratory tests:

1. A two-level system sits in equilibrium with a large heat bath. Are its two states equally
   likely?
2. Grow the bath at a fixed temperature. Does the occupation histogram change its *slope*, or
   only get *cleaner*?
3. Every *joint* microstate of system plus bath is equally probable. Which is more probable:
   the system excited, or in its ground state?
4. Heat the bath without limit. Does the excited population approach $1$, $1/2$ or $0$?

**Your predictions:**

1.
2.
3.
4.

## Part 2 — A composite you can list by hand

Take the two-level system, a bath of three oscillators, and three quanta in all. Before
running the next cell, list on paper every joint microstate: which state the system is in,
and how many quanta each of the three bath oscillators holds. How many are there? In how many
is the system excited?

**Your count:**

*(write here before running the next cell)*

In [ ]:
states = ensembles.explicit_joint_microstates(two_level, n_bath=3, total_quanta=3)
names = ("ground ", "excited")
for level, _, bath in states:
    print(f"system {names[level]}   bath oscillators hold {bath}")

joint = ensembles.enumerate_joint(two_level, n_bath=3, total_quanta=3)
counts = np.rint(np.exp(joint.log_joint_count)).astype(int)
print(f"\n{len(states)} joint microstates; the formula counts {counts[0]} + {counts[1]}")
print(f"P(excited) = {counts[1]}/{counts.sum()} = {ensembles.marginal_occupation(joint)[1]:.4f}")

Each line above is one joint microstate, and each has probability $1/16$ — that is the
postulate, nothing more. When the system is excited it has taken a quantum, and the bath can
hold the two that are left in only $\binom{4}{2} = 6$ ways; in the ground state the bath keeps
all three, in $\binom{5}{3} = 10$ ways. The system's states are weighted by what they leave the
bath.

## Part 3 — Flat joint, curved marginal

The same argument, turned into an assertion. On a bath of six oscillators with six quanta,
every joint microstate has exactly one probability, and the system's own two states do not
share it.

In [ ]:
joint6 = ensembles.enumerate_joint(two_level, n_bath=6, total_quanta=6)
listed = ensembles.explicit_joint_microstates(two_level, n_bath=6, total_quanta=6)
per_joint = np.full(len(listed), joint6.joint_microstate_probability)
marginal6 = ensembles.marginal_occupation(joint6)

print(f"{len(listed)} joint microstates, each with probability 1/{len(listed)}")
print(f"distinct joint probabilities: {np.unique(per_joint).size}")
print(f"system marginal: P(ground) = {marginal6[0]:.4f},  P(excited) = {marginal6[1]:.4f}")

# The falsifying experiment for "the canonical ensemble means every microstate is equally
# probable": one probability for every joint microstate, two for the system's own states.
assert np.ptp(per_joint) == 0.0
assert abs(per_joint.sum() - 1.0) < 1e-12
assert marginal6[1] < marginal6[0]

## Part 4 — Growing the bath at a fixed temperature

Keep one quantum per bath oscillator, which fixes the bath's temperature, and grow the bath.
Watch the three-oscillator solid's level populations against the Boltzmann values
$g_k e^{-E_k/(\kB T)}/Z$ of an infinitely large bath at that temperature.

In [ ]:
temperature = ensembles.bath_temperature(1.0, QUANTUM)
reference = ensembles.boltzmann_distribution(solid, temperature)
sizes = [3, 10, 30, 100, 1000]
sweep = ensembles.bath_size_sweep(solid, sizes, quanta_per_oscillator=1.0)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
colours = plt.cm.viridis(np.linspace(0.0, 0.85, len(sizes)))
k = solid.quanta[:15]
for n_bath, marginal, colour in zip(sweep.n_bath, sweep.marginals, colours, strict=True):
    left.plot(k, marginal[:15], "o-", color=colour, ms=4, label=f"N_bath = {n_bath}")
    reach = marginal[:15] > 0
    right.plot(k[reach], np.log(marginal[:15][reach] / solid.degeneracy[:15][reach]), "o-",
               color=colour, ms=4)
left.plot(k, reference[:15], "k--", lw=2, label="Boltzmann")
right.plot(k, np.log(reference[:15] / solid.degeneracy[:15]), "k--", lw=2)
left.set_xlabel("E_k / quantum")
left.set_ylabel("P_k")
left.legend(fontsize=8)
right.set_xlabel("E_k / quantum")
right.set_ylabel("ln(P_k / g_k)")
right.set_title("log probability of one system microstate")
plt.tight_layout()
plt.show()

for n_bath, distance in zip(sweep.n_bath, sweep.distances, strict=True):
    print(f"N_bath = {n_bath:5d}   largest gap to Boltzmann = {distance:.2e}")

The slope of the right-hand panel is $-\beta\varepsilon$ and it does not change as the bath
grows — the bath's energy per oscillator sets it. What changes is the *bend*: a small bath's
points curve downward, because each quantum the system takes cools the bath a little and makes
the next one dearer. A big bath does not notice.

## Part 5 — How fast does the finite bath converge?

### Predict

The derivation on the module page drops a term of order $E^2/(N_{\mathrm{bath}}(\kB T)^2)$.
If you double the bath at fixed temperature, what should happen to the largest gap between the
exact marginal and the Boltzmann distribution?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
bath_sizes = [20, 40, 80, 160, 320, 640]
scaling = {}
for label, levels in (("two-level", two_level), ("3-oscillator solid", solid)):
    run = ensembles.bath_size_sweep(levels, bath_sizes, quanta_per_oscillator=1.0)
    (slope, _), cov = np.polyfit(np.log(run.n_bath), np.log(run.distances), 1, cov=True)
    scaling[label] = (run, slope, float(np.sqrt(cov[0, 0])))

fig, ax = plt.subplots(figsize=(6.5, 4))
for (label, (run, _, _)), colour in zip(scaling.items(), ("#2563eb", "#d97706"),
                                                strict=True):
    ax.loglog(run.n_bath, run.distances, "o", color=colour, label=label)
    ax.loglog(run.n_bath, run.distances[0] * run.n_bath[0] / run.n_bath, color=colour, lw=1)
ax.set_xlabel("N_bath")
ax.set_ylabel("max_k |P_k - P_k(Boltzmann)|")
ax.legend()
plt.tight_layout()
plt.show()

for label, (run, slope, error) in scaling.items():
    halvings = run.distances[:-1] / run.distances[1:]
    print(f"{label:<20} exponent alpha = {slope:.4f} +/- {error:.4f}   "
          f"(each doubling divides the gap by {halvings.min():.3f} to {halvings.max():.3f})")

The module page predicts the dropped term exactly. Beyond $-\beta E_s$, $\ln P_s$ gains

$$
\frac{E_s^2}{2} \, \frac{\partial^2 \ln \Omega_{\mathrm{bath}}}{\partial U^2}
= -\frac{E_s^2}{2 \kB T^2 \, C_{\mathrm{bath}}} .
$$

The exact count *measures* the same thing, level by level, as

$$
\ln \frac{\Omega_{\mathrm{bath}}(q - k)}{\Omega_{\mathrm{bath}}(q)} + \beta E_k .
$$

Compare them.

In [ ]:
small_solid = ensembles.einstein_levels(3, 6, QUANTUM)
print(" N_bath    k   measured     predicted    ratio")
for n_bath in (30, 100, 300, 1000):
    joint_n = ensembles.enumerate_joint(small_solid, n_bath, n_bath)
    measured = ensembles.measured_log_correction(joint_n)
    predicted = ensembles.predicted_log_correction(joint_n)
    for level in (1, 4):
        print(f"{n_bath:6d}   {level:2d}   {measured[level]:+.5f}    {predicted[level]:+.5f}"
              f"    {measured[level] / predicted[level]:.4f}")

The ratio approaches $1$ as the bath grows, and its distance from $1$ is itself about
$E_k/U_{\mathrm{bath}}$ — the next term of the same expansion. "An infinite bath" is an
approximation whose error you have now measured twice: once as the gap between distributions,
falling as $1/N_{\mathrm{bath}}$, and once term by term.

## Part 6 — Reading the temperature off the bath

Module 09 defined temperature by $1/T = \partial S/\partial U$. With $S = \kB\ln\Omega$, the
bath's $\beta = 1/(\kB T)$ is the slope of its $\ln\Omega$. Measure it the way you would with
nothing but the counts: add a quantum, remove a quantum, and divide by two quanta. Compare
with the exact slope, $\beta\varepsilon = \sum_{j=1}^{N-1} 1/(q + j)$.

In [ ]:
print(" N_bath   beta (counted) [1/J]   beta (exact) [1/J]    relative gap   gap to ln2/quantum")
counted_gaps = []
for n_bath in (10, 20, 40, 80, 160, 320, 640):
    counted = ensembles.beta_of_bath(n_bath, n_bath, QUANTUM)
    exact = ensembles.bath_beta_exact(n_bath, n_bath, QUANTUM)
    counted_gaps.append(relative_error(counted, exact))
    print(f"{n_bath:6d}   {counted:.8e}      {exact:.8e}     {counted_gaps[-1]:.2e}"
          f"       {relative_error(exact, np.log(2) / QUANTUM):.2e}")

beta_order = -scaling_exponent([10, 20, 40, 80, 160, 320, 640], counted_gaps)
print(f"\nthe counted beta converges on the exact slope at order {beta_order:.2f} in 1/N_bath")
print(f"one oscillator alone: beta = {ensembles.bath_beta_exact(1, 5, QUANTUM)} "
      "(it holds any number of quanta in exactly one way)")

## Part 7 — Whose temperature is it?

### Predict

Three baths each hold exactly $100$ quanta, but they have $25$, $100$ and $400$ oscillators.
The same two-level system is put on each in turn. The baths have the same energy. Will the
system's occupations be the same on all three?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
print(" N_bath   bath T [K]   P(excited)   Boltzmann at that T")
equal_energy = {}
for n_bath in (25, 100, 400):
    p = ensembles.marginal_occupation(ensembles.enumerate_joint(two_level, n_bath, 100))
    bath_t = 1.0 / (K_B * ensembles.bath_beta_exact(n_bath, 100, QUANTUM))
    boltzmann = ensembles.boltzmann_distribution(two_level, bath_t)[1]
    equal_energy[n_bath] = (p[1], boltzmann)
    print(f"{n_bath:6d}   {bath_t:9.1f}    {p[1]:.4f}       {boltzmann:.4f}")

Same energy, three different answers. The occupation follows the bath's *slope*, not its
energy, and each value matches the Boltzmann factor at that bath's own temperature. The system
brings no temperature to the contact; it inherits one. That is why a thermometer reads the
temperature of whatever it touches.

## Part 8 — The two-level system, in closed form

For a bath large enough to be treated as infinite, the two-level system has
$p_1 = 1/(e^{\Delta/(\kB T)} + 1)$ and $\langle E \rangle = \Delta\,p_1$.

In [ ]:
system = ensembles.TwoLevelSystem(gap=QUANTUM)
x = np.geomspace(0.05, 100.0, 300)  # k_B T / gap
temperatures = x * QUANTUM / K_B

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
left.semilogx(x, system.ground_occupation(temperatures), color="#2563eb", label="p_0")
left.semilogx(x, system.excited_occupation(temperatures), color="#dc2626", label="p_1")
left.axhline(0.5, color="grey", ls="--", lw=1)
left.set_xlabel("k_B T / gap")
left.set_ylabel("occupation")
left.legend()
right.semilogx(x, system.mean_energy(temperatures) / QUANTUM, color="black")
right.axhline(0.5, color="grey", ls="--", lw=1)
right.set_xlabel("k_B T / gap")
right.set_ylabel("<E> / gap")
plt.tight_layout()
plt.show()

hottest = float(system.excited_occupation(temperatures[-1]))
print(f"coldest shown:  p_1 = {float(system.excited_occupation(temperatures[0])):.2e}")
print(f"hottest shown:  p_1 = {hottest:.6f}   <E>/gap = {hottest:.6f}")
print(f"largest p_1 anywhere on the curve: {np.max(system.excited_occupation(temperatures)):.6f}")

## Part 9 — Optional coda: the atmosphere as a Boltzmann factor

A molecule at height $h$ has energy $m g h$. Treat the rest of the atmosphere as its bath and
the density — and with it the pressure — falls as $e^{-m g h/(\kB T)}$, a scale height
$H = \kB T/(m g)$.

`data/11-standard-atmosphere.csv` is the International Standard Atmosphere, read its header for
exactly what it is: a reference model, isothermal at $216.65\ \mathrm{K}$ between $11$ and
$20\ \mathrm{km}$ and cooling steadily below. Fit the isothermal layer.

In [ ]:
from pathlib import Path

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    csv_path = Path("..", "..", "..", "data", "11-standard-atmosphere.csv")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSV co-located here.
    csv_path = Path("data", "11-standard-atmosphere.csv")

altitude, air_t, air_p = np.genfromtxt(csv_path, delimiter=",", comments="#").T
upper = altitude >= 11_000
(h_slope, h_intercept), h_cov = np.polyfit(altitude[upper], np.log(air_p[upper]), 1, cov=True)
scale_height = -1.0 / h_slope
scale_error = np.sqrt(h_cov[0, 0]) / h_slope**2
g = 9.80665
molecule_mass = K_B * air_t[upper].mean() / (g * scale_height)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(altitude / 1e3, np.log(air_p), "o", color="#2563eb", ms=4, label="standard atmosphere")
ax.plot(altitude / 1e3, h_intercept + h_slope * altitude, color="#dc2626",
        label="isothermal fit, 11-20 km")
ax.set_xlabel("h (km)")
ax.set_ylabel("ln(P / Pa)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"scale height of the isothermal layer: H = {scale_height:.1f} +/- {scale_error:.1e} m")
print("  (the error bar is only the table's rounding: a model atmosphere is exactly exponential)")
print(f"mean molecular mass from H: {molecule_mass / AMU:.3f} u  (air: 28.96 u)")
print(f"nitrogen at 288 K would have H = {K_B * 288 / (28.0134 * AMU * g) / 1e3:.2f} km")

Above $11\ \mathrm{km}$ the points lie on the line, and its slope gives back the molecular mass
of air — a microscopic quantity from a pressure profile. Below $11\ \mathrm{km}$ they bend away:
the air there is warmer, its scale height longer, and a single Boltzmann factor at one
temperature does not describe a layer that is not at one temperature. The failure is the
isothermal assumption, not the Boltzmann factor. (Remember what this table is: fitting it
recovers the standard's own inputs. It tests your reading of the physics, not the sky.)

## Part 10 — Automated checks

A calculation you have not checked is a picture, not evidence. These are the same assertions
that run in the project's test suite.

In [ ]:
# 1. The hand count (Part 2): 16 joint microstates, 10 + 6, P(excited) = 6/16.
assert len(states) == 16 and list(counts) == [10, 6]
assert abs(ensembles.marginal_occupation(joint)[1] - 6 / 16) < 1e-14

# 2. The joint distribution is flat and the marginal is not (Part 3) -- checked above.

# 3. A large bath reproduces the Boltzmann distribution (Part 4) ...
assert sweep.distances[-1] < 2e-3
# ... and the finite-bath gap falls as 1/N_bath for both systems (Part 5).
for _, slope, _ in scaling.values():
    assert abs(slope + 1.0) < 0.02

# 4. The counted beta converges on the exact slope at second order (Part 6).
assert 1.95 < beta_order < 2.05

# 5. Equal energy, different baths, different occupations -- each Boltzmann at its own
#    temperature (Part 7).
occupations = [value[0] for value in equal_energy.values()]
assert occupations[0] > occupations[1] > occupations[2]
for exact_p, boltzmann_p in equal_energy.values():
    assert relative_error(exact_p, boltzmann_p) < 5e-3

# 6. The two-level system never inverts (Part 8).
assert np.all(system.excited_occupation(temperatures) < 0.5)

# 7. The isothermal layer's scale height gives back the mass of an air molecule (Part 9).
assert relative_error(molecule_mass / AMU, 28.9644) < 2e-3

print("all checks passed")

## Part 11 — Explore it yourself

Choose the small system, its gap, the bath's size and how many quanta each bath oscillator
holds on average, then press **Run Interact**. The left panel shows the exact populations
against the Boltzmann values; the right panel shows the log probability of one system
microstate. Three things worth trying:

1. A bath of two or three oscillators at a low energy per oscillator. How far from the
   Boltzmann values can the populations be?
2. A gap of five quanta on a bath with a tenth of a quantum per oscillator. Is the excited
   state reachable at all on a small bath?
3. Any setting at all: can you make the upper level of a two-level system the more populated?

In [ ]:
import ipywidgets as widgets


def explore(system_kind="two-level", gap_quanta=1, bath_oscillators=30, per_oscillator=1.0):
    levels = (ensembles.two_level(gap_quanta, QUANTUM) if system_kind == "two-level"
              else ensembles.einstein_levels(3, 30, QUANTUM))
    total = int(round(per_oscillator * bath_oscillators))
    joint_x = ensembles.enumerate_joint(levels, bath_oscillators, total)
    exact = ensembles.marginal_occupation(joint_x)
    boltzmann = ensembles.boltzmann_distribution(
        levels, ensembles.bath_temperature(per_oscillator, QUANTUM))
    shown = slice(0, 16)
    k = levels.quanta[shown]
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
    left.bar(k, exact[shown], width=0.6, color="#2563eb", label="exact, finite bath")
    left.plot(k, boltzmann[shown], "ko", label="Boltzmann")
    left.set_xlabel("E / quantum")
    left.set_ylabel("P")
    left.legend()
    reach = exact[shown] > 0
    right.plot(k[reach], np.log(exact[shown][reach] / levels.degeneracy[shown][reach]), "o-",
               color="#2563eb")
    right.plot(k, np.log(boltzmann[shown] / levels.degeneracy[shown]), "k--")
    right.set_xlabel("E / quantum")
    right.set_ylabel("ln(P / g)")
    plt.tight_layout()
    plt.show()
    print(f"q_tot = {total} quanta;  largest gap to Boltzmann = "
          f"{ensembles.sup_norm(exact, boltzmann):.2e}")


widgets.interact_manual(
    explore,
    system_kind=["two-level", "3-oscillator solid"],
    gap_quanta=widgets.IntSlider(value=1, min=1, max=5),
    bath_oscillators=widgets.IntSlider(value=30, min=2, max=500),
    per_oscillator=widgets.FloatLogSlider(value=1.0, base=10, min=-1, max=1, step=0.05),
)

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "11-ensembles.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. In Part 3 every joint microstate had the same probability and the system's two states did
   not. Explain, without equations, how both can be true.
3. Part 7's three baths had the same energy. Say exactly which property of a bath the system
   responds to, and why.
4. Nothing in this laboratory moved, collided or used a random number. What, then, produced
   the Boltzmann factor?

**Your answers:**

1.
2.
3.
4.